# Notebook 1: Data Pipeline & Feature Engineering
Merges all Home Credit files into two feature sets: thick-file (with history) and thin-file (without).

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR = './data/'   # change to your path

app       = pd.read_csv(DATA_DIR + 'application_train.csv')
bureau    = pd.read_csv(DATA_DIR + 'bureau.csv')
prev      = pd.read_csv(DATA_DIR + 'previous_application.csv')
inst      = pd.read_csv(DATA_DIR + 'installments_payments.csv')
cc        = pd.read_csv(DATA_DIR + 'credit_card_balance.csv')

print('app:', app.shape, '| bureau:', bureau.shape,
      '| prev:', prev.shape, '| inst:', inst.shape, '| cc:', cc.shape)

app: (307511, 122) | bureau: (1716428, 17) | prev: (1670214, 37) | inst: (13605401, 8) | cc: (3840312, 23)


## 1. Bureau Aggregation

In [2]:
bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    bureau_count          = ('SK_ID_BUREAU',        'count'),
    bureau_active_count   = ('CREDIT_ACTIVE',        lambda x: (x == 'Active').sum()),
    bureau_closed_count   = ('CREDIT_ACTIVE',        lambda x: (x == 'Closed').sum()),
    bureau_avg_overdue    = ('CREDIT_DAY_OVERDUE',   'mean'),
    bureau_max_overdue    = ('CREDIT_DAY_OVERDUE',   'max'),
    bureau_total_debt     = ('AMT_CREDIT_SUM_DEBT',  'sum'),
    bureau_avg_credit     = ('AMT_CREDIT_SUM',       'mean'),
    bureau_prolong_count  = ('CNT_CREDIT_PROLONG',   'sum'),
).reset_index()

# Derived: overdue ratio (proxy for past repayment discipline)
bureau_agg['bureau_overdue_ratio'] = (
    bureau_agg['bureau_avg_overdue'] / (bureau_agg['bureau_avg_credit'] + 1)
)
print(bureau_agg.shape)

(305811, 10)


## 2. Previous Application Aggregation

In [3]:
prev_agg = prev.groupby('SK_ID_CURR').agg(
    prev_app_count        = ('SK_ID_PREV',           'count'),
    prev_approved_count   = ('NAME_CONTRACT_STATUS',  lambda x: (x == 'Approved').sum()),
    prev_refused_count    = ('NAME_CONTRACT_STATUS',  lambda x: (x == 'Refused').sum()),
    prev_avg_credit       = ('AMT_CREDIT',            'mean'),
    prev_avg_annuity      = ('AMT_ANNUITY',           'mean'),
    prev_avg_down_payment = ('AMT_DOWN_PAYMENT',      'mean'),
).reset_index()

prev_agg['prev_approval_rate'] = (
    prev_agg['prev_approved_count'] / (prev_agg['prev_app_count'] + 1e-5)
)
print(prev_agg.shape)

(338857, 8)


## 3. Installments Aggregation (Payment Behavior)

In [4]:
inst['DAYS_LATE']         = inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']
inst['PAYMENT_RATIO']     = inst['AMT_PAYMENT'] / (inst['AMT_INSTALMENT'] + 1e-5)
inst['IS_LATE']           = (inst['DAYS_LATE'] > 0).astype(int)

inst_agg = inst.groupby('SK_ID_CURR').agg(
    inst_count            = ('SK_ID_PREV',   'count'),
    inst_avg_days_late    = ('DAYS_LATE',     'mean'),
    inst_max_days_late    = ('DAYS_LATE',     'max'),
    inst_late_rate        = ('IS_LATE',       'mean'),
    inst_avg_pay_ratio    = ('PAYMENT_RATIO', 'mean'),
    inst_min_pay_ratio    = ('PAYMENT_RATIO', 'min'),
).reset_index()
print(inst_agg.shape)

(339587, 7)


## 4. Credit Card Aggregation (Utilization)

In [5]:
cc['UTILIZATION'] = cc['AMT_BALANCE'] / (cc['AMT_CREDIT_LIMIT_ACTUAL'] + 1e-5)

cc_agg = cc.groupby('SK_ID_CURR').agg(
    cc_months_active      = ('MONTHS_BALANCE',  'count'),
    cc_avg_balance        = ('AMT_BALANCE',      'mean'),
    cc_max_balance        = ('AMT_BALANCE',      'max'),
    cc_avg_utilization    = ('UTILIZATION',      'mean'),
    cc_max_utilization    = ('UTILIZATION',      'max'),
    cc_avg_drawings       = ('AMT_DRAWINGS_CURRENT', 'mean'),
).reset_index()
print(cc_agg.shape)

(103558, 7)


## 5. Base Feature Engineering on application_train

In [6]:
# Core demographic + financial features only (no EXT_SOURCE)
BASE_COLS = [
    'SK_ID_CURR', 'TARGET',
    'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
    'NAME_INCOME_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE',
    'DAYS_EMPLOYED', 'DAYS_BIRTH',
    'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
    'NAME_HOUSING_TYPE',
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'REGION_POPULATION_RELATIVE', 'REGION_RATING_CLIENT',
]

df = app[BASE_COLS].copy()

# ── Engineered ratios ──────────────────────────────────────────────────────
df['AGE_YEARS']              = -df['DAYS_BIRTH'] / 365
df['EMPLOYED_YEARS']         = df['DAYS_EMPLOYED'].apply(lambda x: -x/365 if x < 0 else 0)
df['DEBT_TO_INCOME']         = df['AMT_CREDIT']   / (df['AMT_INCOME_TOTAL'] + 1)
df['ANNUITY_TO_INCOME']      = df['AMT_ANNUITY']  / (df['AMT_INCOME_TOTAL'] + 1)
df['CREDIT_TO_GOODS']        = df['AMT_CREDIT']   / (df['AMT_GOODS_PRICE']  + 1)
df['INCOME_PER_PERSON']      = df['AMT_INCOME_TOTAL'] / (df['CNT_FAM_MEMBERS'] + 1)
df['CHILDREN_RATIO']         = df['CNT_CHILDREN'] / (df['CNT_FAM_MEMBERS'] + 1)
df['EMI_BURDEN']             = df['AMT_ANNUITY']  / (df['AMT_INCOME_TOTAL'] + 1)
df['INCOME_STABILITY']       = df['EMPLOYED_YEARS'] / (df['AGE_YEARS'] + 1)
df['LOAN_TO_INCOME']         = df['AMT_CREDIT']   / (df['AMT_INCOME_TOTAL'] * 12 + 1)

# Flag OWN as numeric
df['FLAG_OWN_CAR']    = (df['FLAG_OWN_CAR']    == 'Y').astype(int)
df['FLAG_OWN_REALTY'] = (df['FLAG_OWN_REALTY'] == 'Y').astype(int)

# Drop raw date cols
df.drop(columns=['DAYS_BIRTH', 'DAYS_EMPLOYED'], inplace=True)
print('Base df shape:', df.shape)

Base df shape: (307511, 28)


## 6. Full Merge → Thick-File Dataset

In [7]:
thick = (
    df
    .merge(bureau_agg, on='SK_ID_CURR', how='left')
    .merge(prev_agg,   on='SK_ID_CURR', how='left')
    .merge(inst_agg,   on='SK_ID_CURR', how='left')
    .merge(cc_agg,     on='SK_ID_CURR', how='left')
)

# ── Thin-file flag: no bureau + no installment + no prev application ───────
thick['IS_THIN_FILE'] = (
    thick['bureau_count'].isna() &
    thick['inst_count'].isna()   &
    thick['prev_app_count'].isna()
).astype(int)

print('Thick-file dataset shape:', thick.shape)
print('Thin-file users:', thick['IS_THIN_FILE'].sum(),
      f'({thick["IS_THIN_FILE"].mean()*100:.1f}%)')

Thick-file dataset shape: (307511, 57)
Thin-file users: 2240 (0.7%)


## 7. Categorical Encoding + Save

In [8]:
from sklearn.preprocessing import LabelEncoder
import pickle

CAT_COLS = [
    'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_INCOME_TYPE',
    'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'NAME_HOUSING_TYPE'
]

encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    thick[col] = thick[col].fillna('Unknown')
    thick[col] = le.fit_transform(thick[col])
    encoders[col] = le

# Fill numeric NaNs with 0 (absence of record = 0 activity)
numeric_cols = thick.select_dtypes(include=[np.number]).columns
thick[numeric_cols] = thick[numeric_cols].fillna(0)

# Save
thick.to_parquet('thick_file_dataset.parquet', index=False)
with open('label_encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

print('Saved: thick_file_dataset.parquet')
print('Class balance:\n', thick['TARGET'].value_counts(normalize=True).round(3))

Saved: thick_file_dataset.parquet
Class balance:
 TARGET
0    0.919
1    0.081
Name: proportion, dtype: float64


## 8. Define Feature Sets for Notebooks 2 & 3

In [9]:
# TEACHER features: everything except ID, TARGET, IS_THIN_FILE
TEACHER_FEATURES = [c for c in thick.columns
                    if c not in ('SK_ID_CURR', 'TARGET', 'IS_THIN_FILE')]

# STUDENT features: only non-credit-history cols
HISTORY_COLS = [
    'bureau_count', 'bureau_active_count', 'bureau_closed_count',
    'bureau_avg_overdue', 'bureau_max_overdue', 'bureau_total_debt',
    'bureau_avg_credit', 'bureau_prolong_count', 'bureau_overdue_ratio',
    'prev_app_count', 'prev_approved_count', 'prev_refused_count',
    'prev_avg_credit', 'prev_avg_annuity', 'prev_avg_down_payment',
    'prev_approval_rate',
    'inst_count', 'inst_avg_days_late', 'inst_max_days_late',
    'inst_late_rate', 'inst_avg_pay_ratio', 'inst_min_pay_ratio',
    'cc_months_active', 'cc_avg_balance', 'cc_max_balance',
    'cc_avg_utilization', 'cc_max_utilization', 'cc_avg_drawings',
]

STUDENT_FEATURES = [c for c in TEACHER_FEATURES if c not in HISTORY_COLS]

import json
with open('feature_sets.json', 'w') as f:
    json.dump({'teacher': TEACHER_FEATURES, 'student': STUDENT_FEATURES,
               'history_cols': HISTORY_COLS}, f)

print(f'Teacher features: {len(TEACHER_FEATURES)}')
print(f'Student features: {len(STUDENT_FEATURES)}')
print('\nStudent features:', STUDENT_FEATURES)

Teacher features: 54
Student features: 26

Student features: ['CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_INCOME_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_HOUSING_TYPE', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'REGION_POPULATION_RELATIVE', 'REGION_RATING_CLIENT', 'AGE_YEARS', 'EMPLOYED_YEARS', 'DEBT_TO_INCOME', 'ANNUITY_TO_INCOME', 'CREDIT_TO_GOODS', 'INCOME_PER_PERSON', 'CHILDREN_RATIO', 'EMI_BURDEN', 'INCOME_STABILITY', 'LOAN_TO_INCOME']
